<a href="https://colab.research.google.com/github/pxs1990/NLP_LLM/blob/main/peft_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! pip install -U transformers datasets peft accelerate scikit-learn pandas torch
! pip install -U bitsandbytes # Optional for low VRAM:

In [ ]:
import os, pandas as pd, numpy as np
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
import torch
from transformers import (
     AutoTokenizer,
     AutoModelForSequenceClassification,
     DataCollatorWithPadding,
     TrainingArguments,
     Trainer,
     )
from peft import LoraConfig, get_peft_model

In [ ]:
MODEL_NAME = "bert-base-uncased"  # any BERT-like model

In [ ]:
from sklearn.preprocessing import LabelEncoder

# ========= 1) Load & prepare data =========
df = pd.read_csv("train.csv")  # expects 'text','label'
assert {"text", "label"}.issubset(df.columns), "CSV must have text,label columns"

# --- Safety checks ---
if df["label"].isna().any():
    raise ValueError("Label column contains NaN values")

# ========= 2) Encode labels (0..N-1) =========
le = LabelEncoder()
df["label"] = le.fit_transform(df["label"])

# Build mappings (Hugging Face expects these)
label2id = {label: i for i, label in enumerate(le.classes_)}
id2label = {i: str(label) for label, i in label2id.items()}
num_labels = len(le.classes_)

# ========= 3) Train / validation split =========
train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    stratify=df["label"],
    random_state=42
)

# ========= 4) Convert to Hugging Face datasets =========
ds = DatasetDict({
    "train": Dataset.from_pandas(train_df, preserve_index=False),
    "validation": Dataset.from_pandas(val_df, preserve_index=False),
})


In [ ]:
# ========= 2) Tokenizer =========
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def preprocess(row):
	return tokenizer(row["text"], truncation=True)
keep_cols = ["label"]
ds = ds.map(preprocess, batched=True, remove_columns=[c for c in ds["train"].column_names if c not in keep_cols])

collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [ ]:
# ========= 3) Base model =========
# (Optional low-VRAM: load in 8-bit with bitsandbytes; uncomment below)
# from transformers import BitsAndBytesConfig
# quant_cfg = BitsAndBytesConfig(load_in_8bit=True)
# base_model = AutoModelForSequenceClassification.from_pretrained(
#     MODEL_NAME, num_labels=num_labels, id2label=id2label, label2id={v:k for k,v in id2label.items()},
#     quantization_config=quant_cfg, device_map="auto"
# )

base_model = AutoModelForSequenceClassification.from_pretrained(
	MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id={v: k for k, v in id2label.items()},
)

In [ ]:
# ========= 4) Apply LoRA adapters with PEFT =========
# For BERT, common target module substrings: "query","key","value","dense"
lora_cfg = LoraConfig(
	r=8,
    lora_alpha=16,
    target_modules=["query","key","value","dense"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS",
)
model = get_peft_model(base_model, lora_cfg)
model.print_trainable_parameters()  # sanity check


In [ ]:
# ========= 5) Def Compute Metrics =========
def compute_metrics(p):
	preds = np.argmax(p.predictions, axis=1)
	return {
        "accuracy": accuracy_score(p.label_ids, preds),
        "f1_macro": f1_score(p.label_ids, preds, average="macro"),
	}


In [ ]:
# ========= 6) Def Training args & Trainer =========
args = TrainingArguments(
    output_dir="bert-lora-multiclass",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    learning_rate=2e-4,        	# LoRA can use a higher LR than full fine-tuning
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_strategy="steps",
    logging_steps=50,
    gradient_accumulation_steps=1,
    fp16=torch.cuda.is_available(),   # enable mixed precision if on GPU
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
	args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
Project structure
hier-bert/
  README.md
  requirements.txt
  configs/
    config.yaml
  data/
    train.csv
  src/
    train.py
    model.py
    data.py
    metrics.py
    infer.py
    utils.py

Data format (data/train.csv)

Required columns:

text (string)

level1 (parent label)

level2 (child label)

Example:

text,level1,level2
"Premier League match report",Sports,Football
"NBA finals analysis",Sports,Basketball
"New LLM model release",Tech,AI

requirements.txt
transformers>=4.40.0
datasets>=2.18.0
torch>=2.1.0
scikit-learn>=1.3.0
pandas>=2.0.0
pyyaml>=6.0.0
joblib>=1.3.0

configs/config.yaml
model_name: bert-base-uncased
max_length: 256

test_size: 0.1
random_state: 42

train:
  output_dir: outputs/hier_bert
  per_device_train_batch_size: 16
  per_device_eval_batch_size: 32
  learning_rate: 2.0e-5
  num_train_epochs: 3
  weight_decay: 0.01
  warmup_ratio: 0.06
  logging_steps: 50
  eval_strategy: epoch
  save_strategy: epoch
  load_best_model_at_end: true
  metric_for_best_model: eval_level2_f1_macro
  greater_is_better: true

loss_weights:
  level1: 1.0
  level2: 1.0

hierarchy:
  enforce_constraint_at_inference: true

src/utils.py
import json
from pathlib import Path
import joblib

def ensure_dir(path: str) -> None:
    Path(path).mkdir(parents=True, exist_ok=True)

def save_json(obj, path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def load_json(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_joblib(obj, path: str) -> None:
    joblib.dump(obj, path)

def load_joblib(path: str):
    return joblib.load(path)

src/data.py
from dataclasses import dataclass
from typing import Dict, Tuple

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset, DatasetDict

@dataclass
class Encoders:
    level1: LabelEncoder
    level2: LabelEncoder

def build_hierarchy_map(df: pd.DataFrame, le_l1: LabelEncoder, le_l2: LabelEncoder) -> Dict[int, list]:
    """
    Returns: {level1_id: [allowed level2_id, ...], ...}
    """
    mapping = {}
    for l1, sub in df.groupby("level1"):
        l1_id = int(le_l1.transform([l1])[0])
        l2_ids = sorted(set(int(x) for x in le_l2.transform(sub["level2"].astype(str).tolist())))
        mapping[l1_id] = l2_ids
    return mapping

def load_and_prepare(
    csv_path: str,
    test_size: float,
    random_state: int
) -> Tuple[DatasetDict, Encoders, Dict[int, list]]:
    df = pd.read_csv(csv_path)
    required = {"text", "level1", "level2"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    # Safety checks
    if df[["text", "level1", "level2"]].isna().any().any():
        raise ValueError("Found NaN in text/level1/level2. Clean the data first.")

    # Fit encoders on full data (OK if you treat this as 'training universe').
    # If you want strictest evaluation, fit on train only, then transform val.
    le_l1 = LabelEncoder()
    le_l2 = LabelEncoder()

    df["level1_id"] = le_l1.fit_transform(df["level1"].astype(str))
    df["level2_id"] = le_l2.fit_transform(df["level2"].astype(str))

    hierarchy_map = build_hierarchy_map(df, le_l1, le_l2)

    # Stratify by the most specific label to keep distribution
    train_df, val_df = train_test_split(
        df,
        test_size=test_size,
        stratify=df["level2_id"],
        random_state=random_state
    )

    ds = DatasetDict({
        "train": Dataset.from_pandas(train_df[["text", "level1_id", "level2_id"]], preserve_index=False),
        "validation": Dataset.from_pandas(val_df[["text", "level1_id", "level2_id"]], preserve_index=False),
    })

    return ds, Encoders(level1=le_l1, level2=le_l2), hierarchy_map

src/model.py
from dataclasses import dataclass
from typing import Optional, Dict, Any

import torch
import torch.nn as nn
from transformers import AutoModel, PreTrainedModel, PretrainedConfig

@dataclass
class HierConfig:
    base_model_name: str
    num_level1: int
    num_level2: int
    dropout: float = 0.1

class HierBert(nn.Module):
    """
    BERT encoder + two classification heads.
    """
    def __init__(self, model_name: str, num_level1: int, num_level2: int, dropout: float = 0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.head_l1 = nn.Linear(hidden, num_level1)
        self.head_l2 = nn.Linear(hidden, num_level2)

    def forward(self, input_ids, attention_mask=None, token_type_ids=None) -> Dict[str, torch.Tensor]:
        out = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        # Use [CLS] pooled rep if available, else use first token
        if hasattr(out, "pooler_output") and out.pooler_output is not None:
            pooled = out.pooler_output
        else:
            pooled = out.last_hidden_state[:, 0]

        x = self.dropout(pooled)
        logits_l1 = self.head_l1(x)
        logits_l2 = self.head_l2(x)
        return {"logits_l1": logits_l1, "logits_l2": logits_l2}

src/metrics.py
from typing import Dict
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_level_metrics(y_true: np.ndarray, y_pred: np.ndarray, prefix: str) -> Dict[str, float]:
    return {
        f"{prefix}_acc": float(accuracy_score(y_true, y_pred)),
        f"{prefix}_f1_macro": float(f1_score(y_true, y_pred, average="macro")),
        f"{prefix}_f1_micro": float(f1_score(y_true, y_pred, average="micro")),
    }

src/train.py
import argparse
import yaml
import numpy as np
import torch
import torch.nn as nn

from transformers import AutoTokenizer, TrainingArguments, Trainer
from datasets import DatasetDict

from src.data import load_and_prepare
from src.model import HierBert
from src.metrics import compute_level_metrics
from src.utils import ensure_dir, save_joblib, save_json

class HierTrainer(Trainer):
    def __init__(self, *args, loss_w_l1=1.0, loss_w_l2=1.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.loss_w_l1 = loss_w_l1
        self.loss_w_l2 = loss_w_l2
        self.ce = nn.CrossEntropyLoss()

    def compute_loss(self, model, inputs, return_outputs=False):
        level1 = inputs.pop("level1_id")
        level2 = inputs.pop("level2_id")

        outputs = model(**inputs)
        logits_l1 = outputs["logits_l1"]
        logits_l2 = outputs["logits_l2"]

        loss_l1 = self.ce(logits_l1, level1)
        loss_l2 = self.ce(logits_l2, level2)
        loss = self.loss_w_l1 * loss_l1 + self.loss_w_l2 * loss_l2

        if return_outputs:
            outputs["loss_l1"] = loss_l1.detach()
            outputs["loss_l2"] = loss_l2.detach()
            return loss, outputs
        return loss

def tokenize_batch(tokenizer, max_length: int, batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=max_length
    )

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", type=str, default="configs/config.yaml")
    ap.add_argument("--csv", type=str, default="data/train.csv")
    args = ap.parse_args()

    cfg = yaml.safe_load(open(args.config, "r", encoding="utf-8"))

    ds, encoders, hierarchy_map = load_and_prepare(
        csv_path=args.csv,
        test_size=float(cfg["test_size"]),
        random_state=int(cfg["random_state"]),
    )

    tokenizer = AutoTokenizer.from_pretrained(cfg["model_name"], use_fast=True)

    ds = ds.map(lambda b: tokenize_batch(tokenizer, int(cfg["max_length"]), b), batched=True)
    ds = ds.remove_columns(["text"])
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "level1_id", "level2_id"])

    num_l1 = len(encoders.level1.classes_)
    num_l2 = len(encoders.level2.classes_)

    model = HierBert(
        model_name=cfg["model_name"],
        num_level1=num_l1,
        num_level2=num_l2,
        dropout=0.1
    )

    out_dir = cfg["train"]["output_dir"]
    ensure_dir(out_dir)

    # Save encoders + hierarchy for inference
    save_joblib(encoders.level1, f"{out_dir}/labelenc_level1.joblib")
    save_joblib(encoders.level2, f"{out_dir}/labelenc_level2.joblib")
    save_json(hierarchy_map, f"{out_dir}/hierarchy_map.json")

    def compute_metrics(eval_pred):
        # eval_pred.predictions will be whatever Trainer returns; we need both heads.
        # We'll pack them from our custom prediction_step behavior by using a custom override:
        raise RuntimeError("compute_metrics is handled by custom evaluate loop below.")

    # --- Custom Trainer: override prediction_step to return both logits ---
    class _HierTrainer(HierTrainer):
        def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
            with torch.no_grad():
                level1 = inputs["level1_id"]
                level2 = inputs["level2_id"]
                outputs = model(
                    input_ids=inputs["input_ids"],
                    attention_mask=inputs.get("attention_mask", None),
                    token_type_ids=inputs.get("token_type_ids", None),
                )
                logits_l1 = outputs["logits_l1"]
                logits_l2 = outputs["logits_l2"]

                loss = None
                if not prediction_loss_only:
                    # Return logits in a tuple, Trainer will stack them
                    return loss, (logits_l1.detach().cpu().numpy(), logits_l2.detach().cpu().numpy()), (
                        level1.detach().cpu().numpy(), level2.detach().cpu().numpy()
                    )
                return loss, None, None

        def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix="eval"):
            eval_dataset = eval_dataset if eval_dataset is not None else self.eval_dataset
            output = super().evaluate(eval_dataset=eval_dataset, ignore_keys=ignore_keys, metric_key_prefix=metric_key_prefix)
            return output

    training_args = TrainingArguments(
        output_dir=out_dir,
        per_device_train_batch_size=int(cfg["train"]["per_device_train_batch_size"]),
        per_device_eval_batch_size=int(cfg["train"]["per_device_eval_batch_size"]),
        learning_rate=float(cfg["train"]["learning_rate"]),
        num_train_epochs=float(cfg["train"]["num_train_epochs"]),
        weight_decay=float(cfg["train"]["weight_decay"]),
        warmup_ratio=float(cfg["train"]["warmup_ratio"]),
        logging_steps=int(cfg["train"]["logging_steps"]),
        eval_strategy=str(cfg["train"]["eval_strategy"]),
        save_strategy=str(cfg["train"]["save_strategy"]),
        load_best_model_at_end=bool(cfg["train"]["load_best_model_at_end"]),
        metric_for_best_model=str(cfg["train"]["metric_for_best_model"]),
        greater_is_better=bool(cfg["train"]["greater_is_better"]),
        fp16=torch.cuda.is_available(),
        report_to=[],
    )

    trainer = _HierTrainer(
        model=model,
        args=training_args,
        train_dataset=ds["train"],
        eval_dataset=ds["validation"],
        tokenizer=tokenizer,
        loss_w_l1=float(cfg["loss_weights"]["level1"]),
        loss_w_l2=float(cfg["loss_weights"]["level2"]),
    )

    trainer.train()

    # ---- Compute metrics after training (clean + explicit) ----
    pred = trainer.predict(ds["validation"])
    # pred.predictions is tuple: (logits_l1, logits_l2)
    logits_l1, logits_l2 = pred.predictions
    y1_true, y2_true = pred.label_ids

    y1_pred = np.argmax(logits_l1, axis=1)
    y2_pred = np.argmax(logits_l2, axis=1)

    metrics = {}
    metrics.update(compute_level_metrics(y1_true, y1_pred, "level1"))
    metrics.update(compute_level_metrics(y2_true, y2_pred, "level2"))

    # Save final metrics
    save_json(metrics, f"{out_dir}/final_metrics.json")
    print("Final metrics:", metrics)

    # Save model + tokenizer
    trainer.save_model(out_dir)
    tokenizer.save_pretrained(out_dir)

if __name__ == "__main__":
    main()

src/infer.py (with hierarchy constraint)
import argparse
import numpy as np
import torch
from transformers import AutoTokenizer
from src.model import HierBert
from src.utils import load_joblib, load_json

@torch.no_grad()
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model_dir", type=str, required=True)
    ap.add_argument("--text", type=str, required=True)
    args = ap.parse_args()

    le_l1 = load_joblib(f"{args.model_dir}/labelenc_level1.joblib")
    le_l2 = load_joblib(f"{args.model_dir}/labelenc_level2.joblib")
    hierarchy_map = load_json(f"{args.model_dir}/hierarchy_map.json")

    tokenizer = AutoTokenizer.from_pretrained(args.model_dir, use_fast=True)

    num_l1 = len(le_l1.classes_)
    num_l2 = len(le_l2.classes_)

    model = HierBert(model_name=args.model_dir, num_level1=num_l1, num_level2=num_l2)
    model.load_state_dict(torch.load(f"{args.model_dir}/pytorch_model.bin", map_location="cpu"))
    model.eval()

    batch = tokenizer(args.text, return_tensors="pt", truncation=True, padding=True, max_length=256)
    out = model(**batch)

    logits_l1 = out["logits_l1"].cpu().numpy()[0]
    logits_l2 = out["logits_l2"].cpu().numpy()[0]

    l1_id = int(np.argmax(logits_l1))
    l1_label = str(le_l1.inverse_transform([l1_id])[0])

    # Optional constraint: allow only children under predicted l1
    allowed = hierarchy_map.get(str(l1_id), hierarchy_map.get(l1_id, None))
    if allowed is not None:
        mask = np.full_like(logits_l2, -1e9, dtype=np.float32)
        for cid in allowed:
            mask[int(cid)] = 0.0
        logits_l2 = logits_l2 + mask

    l2_id = int(np.argmax(logits_l2))
    l2_label = str(le_l2.inverse_transform([l2_id])[0])

    print({"level1_id": l1_id, "level1": l1_label, "level2_id": l2_id, "level2": l2_label})

if __name__ == "__main__":
    main()